In [1]:
!pip install -q transformers==4.38.2 datasets sacrebleu rouge_score accelerate


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
litellm 1.93.2 requires tokenizers<1.0,>=0.21.0, but you have tokenizers 0.15.2 which is incompatible.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.26.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
sentence-transformers 5.7.0 requires tokenizers>=0.19, but you have tokenizers 0.15.2 which is incompatible.
sentence-transformers 5.7.0 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.38.2 which is incompatible.


In [2]:
import transformers
print(transformers.__version__) 

4.38.2


In [3]:

import os, math, warnings, json, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from transformers import (AutoTokenizer, AutoModelForSeq2SeqLM,
                          DataCollatorForSeq2Seq, get_linear_schedule_with_warmup)
from datasets import load_dataset
import sacrebleu
from rouge_score import rouge_scorer
warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

# ── Config ────────────────────────────────────────────────────────────────────
TEACHER_MODEL = "/kaggle/input/datasets/habibajahanpriya/codet5-base-finetuned/finetuned_codet5-base"  
STUDENT_MODEL = "Salesforce/codet5-small"

TEMPERATURE = 4.0
ALPHA       = 0.5      # weight on SOFT loss. Try 0 (baseline), 0.5, 0.7, 0.9

MAX_SOURCE_LEN = 128
MAX_TARGET_LEN = 128
BATCH_SIZE     = 32
EPOCHS         = 10
LR             = 1e-4
WEIGHT_DECAY   = 0.01
WARMUP_RATIO   = 0.1
AMP            = True

SAVE_DIR = f"./best_student_a{ALPHA}_T{TEMPERATURE}_s{SEED}"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

# ── Dataset ───────────────────────────────────────────────────────────────────
dataset = load_dataset("code_x_glue_cc_code_refinement", "small")
# smoke test (uncomment for a quick run)
# dataset["train"]      = dataset["train"].select(range(100))
# dataset["validation"] = dataset["validation"].select(range(100))
# dataset["test"]       = dataset["test"].select(range(100))
print(dataset)

# ── Tokenizers ────────────────────────────────────────────────────────────────
teacher_tokenizer = AutoTokenizer.from_pretrained("Salesforce/codet5-base")
student_tokenizer = AutoTokenizer.from_pretrained(STUDENT_MODEL)
assert teacher_tokenizer.vocab_size == student_tokenizer.vocab_size

# ── Preprocessing ─────────────────────────────────────────────────────────────
def preprocess(examples, tokenizer):
    model_inputs = tokenizer(examples["buggy"], max_length=MAX_SOURCE_LEN,
                             padding=False, truncation=True)
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(examples["fixed"], max_length=MAX_TARGET_LEN,
                           padding=False, truncation=True)
    model_inputs["labels"] = [
        [(l if l != tokenizer.pad_token_id else -100) for l in label]
        for label in labels["input_ids"]
    ]
    return model_inputs

tokenized = dataset.map(lambda x: preprocess(x, student_tokenizer), batched=True,
                        remove_columns=dataset["train"].column_names)
tokenized.set_format("torch")

student_collator = DataCollatorForSeq2Seq(student_tokenizer, padding=True, return_tensors="pt")

g = torch.Generator(); g.manual_seed(SEED)
train_loader = DataLoader(tokenized["train"], batch_size=BATCH_SIZE, collate_fn=student_collator,
                          shuffle=True, num_workers=2, pin_memory=True, generator=g)
val_loader = DataLoader(tokenized["validation"], batch_size=BATCH_SIZE,
                        collate_fn=student_collator, shuffle=False, num_workers=2)

# ── Models ────────────────────────────────────────────────────────────────────
teacher = AutoModelForSeq2SeqLM.from_pretrained(TEACHER_MODEL).to(DEVICE)
student = AutoModelForSeq2SeqLM.from_pretrained(STUDENT_MODEL).to(DEVICE)
teacher.eval()
for p in teacher.parameters():
    p.requires_grad_(False)

param_teacher = sum(p.numel() for p in teacher.parameters()) / 1e6
param_student = sum(p.numel() for p in student.parameters()) / 1e6
print(f"Teacher: {param_teacher:.1f}M | Student: {param_student:.1f}M | "
      f"{param_teacher/param_student:.2f}x")

# ── Distillation loss ─────────────────────────────────────────────────────────
class DistillationLoss(nn.Module):
    def __init__(self, temperature=4.0, alpha=0.5):
        super().__init__()
        self.T, self.alpha = temperature, alpha
        self.kl_div = nn.KLDivLoss(reduction="batchmean")   # = mean per token (masked)

    def forward(self, student_logits, teacher_logits, labels):
        T = self.T
        student_logits = student_logits.float()
        teacher_logits = teacher_logits.float()
        log_p_s = F.log_softmax(student_logits / T, dim=-1)
        p_t     = F.softmax(teacher_logits / T, dim=-1)
        loss_soft = self.kl_div(log_p_s, p_t) * (T * T)
        loss_hard = F.cross_entropy(student_logits, labels, ignore_index=-100)
        total = self.alpha * loss_soft + (1.0 - self.alpha) * loss_hard
        return total, {"hard": loss_hard.item(), "soft": loss_soft.item()}

kd_loss_fn = DistillationLoss(temperature=TEMPERATURE, alpha=ALPHA)

# ── Optimizer / scheduler ─────────────────────────────────────────────────────
optimizer = torch.optim.AdamW(student.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
total_steps  = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, int(total_steps * WARMUP_RATIO), total_steps)
scaler = torch.cuda.amp.GradScaler(enabled=AMP)

def get_seq2seq_logits(model, batch, device):
    labels = batch["labels"].to(device)
    out = model(input_ids=batch["input_ids"].to(device),
                attention_mask=batch["attention_mask"].to(device),
                decoder_input_ids=model._shift_right(labels))
    return out.logits

# ── Training ──────────────────────────────────────────────────────────────────
print(f"\nStarting KD | T={TEMPERATURE} | α={ALPHA} | epochs={EPOCHS} | lr={LR}")
best_val_loss = float("inf")

for epoch in range(1, EPOCHS + 1):
    student.train()
    running = {"loss": 0.0, "hard": 0.0, "soft": 0.0, "n": 0}

    for step, batch in enumerate(train_loader, 1):
        labels = batch["labels"].to(DEVICE)
        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=AMP):
            with torch.no_grad():
                teacher_logits = get_seq2seq_logits(teacher, batch, DEVICE)
            student_logits = get_seq2seq_logits(student, batch, DEVICE)

            B, T_len, V = student_logits.shape
            s_flat = student_logits.reshape(-1, V)
            t_flat = teacher_logits.reshape(-1, V)
            l_flat = labels.reshape(-1)
            mask = l_flat != -100
            total_loss, parts = kd_loss_fn(s_flat[mask], t_flat[mask], l_flat[mask])

        scaler.scale(total_loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(student.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        b = labels.size(0)
        running["loss"] += total_loss.item() * b
        running["hard"] += parts["hard"] * b
        running["soft"] += parts["soft"] * b
        running["n"]    += b
        if step % 200 == 0:
            n = running["n"]
            print(f"  Epoch {epoch} | Step {step}/{len(train_loader)} | "
                  f"loss {running['loss']/n:.4f} "
                  f"(hard {running['hard']/n:.4f}, soft {running['soft']/n:.4f})")

    # Validation: hard CE only (teacher not needed)
    student.eval()
    val_loss, val_tokens = 0.0, 0
    with torch.no_grad():
        for batch in val_loader:
            labels_v = batch["labels"].to(DEVICE)
            with torch.cuda.amp.autocast(enabled=AMP):
                s_v = get_seq2seq_logits(student, batch, DEVICE)
            s_flat = s_v.reshape(-1, s_v.shape[-1]).float()
            l_flat = labels_v.reshape(-1)
            mask = l_flat != -100
            val_loss   += F.cross_entropy(s_flat[mask], l_flat[mask], reduction="sum").item()
            val_tokens += mask.sum().item()
    avg_val = val_loss / val_tokens
    n = running["n"]
    print(f"\nEpoch {epoch:02d} | train_loss {running['loss']/n:.4f} "
          f"(hard {running['hard']/n:.4f}, soft {running['soft']/n:.4f}) | val_CE {avg_val:.4f}")

    if avg_val < best_val_loss:
        best_val_loss = avg_val
        student.save_pretrained(SAVE_DIR)
        student_tokenizer.save_pretrained(SAVE_DIR)
        print(f"  ✓ Saved best model (val_CE={avg_val:.4f})")

print("\nTraining complete. Best val_CE:", best_val_loss)

# ── Evaluation ────────────────────────────────────────────────────────────────
eval_model = AutoModelForSeq2SeqLM.from_pretrained(SAVE_DIR).to(DEVICE)
eval_tokenizer = AutoTokenizer.from_pretrained(SAVE_DIR)
eval_model.eval()

def generate_predictions(model, tokenizer, examples, batch_size=32):
    preds, refs = [], []
    for i in range(0, len(examples), batch_size):
        batch_examples = examples[i:i + batch_size]
        inputs = tokenizer([ex["buggy"] for ex in batch_examples], return_tensors="pt",
                           padding=True, truncation=True, max_length=MAX_SOURCE_LEN).to(DEVICE)
        with torch.no_grad():
            generated_ids = model.generate(input_ids=inputs["input_ids"],
                                           attention_mask=inputs["attention_mask"],
                                           max_length=MAX_TARGET_LEN,
                                           num_beams=4, early_stopping=True)
        preds.extend(tokenizer.batch_decode(
            generated_ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False,   # keeps "VAR_2 ," spacing intact
        ))
        refs.extend(ex["fixed"] for ex in batch_examples)
    return preds, refs

print("Generating predictions on test set...")
test_examples = list(dataset["test"])
predictions, references = generate_predictions(eval_model, eval_tokenizer, test_examples)

with open(f"{SAVE_DIR}_preds.json", "w") as f:
    json.dump({"predictions": predictions, "references": references}, f)

bleu_result = sacrebleu.corpus_bleu(predictions, [references])
scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False)
avg_rouge_l = float(np.mean([scorer.score(r, p)["rougeL"].fmeasure
                             for r, p in zip(references, predictions)]))


print("\nSample predictions (first 3):")
for i in range(3):
    print(f"\n--- Sample {i+1} ---")
    print(f"  Buggy  : {test_examples[i]['buggy'][:120]}")
    print(f"  Fixed  : {references[i][:120]}")
    print(f"  Student: {predictions[i][:120]}")

report = {
    "teacher_model": TEACHER_MODEL, "student_model": STUDENT_MODEL,
    "temperature": TEMPERATURE, "alpha": ALPHA, "epochs": EPOCHS, "lr": LR, "seed": SEED,
    "best_val_ce": best_val_loss,
    "sacrebleu": round(bleu_result.score, 4), "rouge_l": round(avg_rouge_l * 100, 4),
    "teacher_params_M": round(param_teacher, 1), "student_params_M": round(param_student, 1),
    "compression_ratio": round(param_teacher / param_student, 2),
}
with open(f"{SAVE_DIR}_results.json", "w") as f:
    json.dump(report, f, indent=2)
print(json.dumps(report, indent=2))

Using device: cuda


README.md: 0.00B [00:00, ?B/s]

small/train-00000-of-00001.parquet:   0%|          | 0.00/4.72M [00:00<?, ?B/s]

small/validation-00000-of-00001.parquet:   0%|          | 0.00/591k [00:00<?, ?B/s]

small/test-00000-of-00001.parquet:   0%|          | 0.00/589k [00:00<?, ?B/s]

...

Map:   0%|          | 0/5835 [00:00<?, ? examples/s]

Map:   0%|          | 0/5835 [00:00<?, ? examples/s]

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/242M [00:00<?, ?B/s]

Teacher: 222.9M | Student: 60.5M | 3.68x

Starting KD | T=4.0 | α=0.5 | epochs=10 | lr=0.0001
  Epoch 1 | Step 200/1459 | loss 2.7969 (hard 2.2721, soft 3.3218)
  Epoch 1 | Step 400/1459 | loss 1.9092 (hard 1.2731, soft 2.5453)
  Epoch 1 | Step 600/1459 | loss 1.5537 (hard 0.9252, soft 2.1821)
  Epoch 1 | Step 800/1459 | loss 1.3468 (hard 0.7463, soft 1.9474)
  Epoch 1 | Step 1000/1459 | loss 1.2053 (hard 0.6363, soft 1.7743)
  Epoch 1 | Step 1200/1459 | loss 1.0988 (hard 0.5608, soft 1.6368)
  Epoch 1 | Step 1400/1459 | loss 1.0151 (hard 0.5061, soft 1.5242)

Epoch 01 | train_loss 0.9936 (hard 0.4925, soft 1.4947) | val_CE 0.1532
  ✓ Saved best model (val_CE=0.1532)
  Epoch 2 | Step 200/1459 | loss 0.4548 (hard 0.1662, soft 0.7435)
  Epoch 2 | Step 400/1459 | loss 0.4412 (hard 0.1655, soft 0.7168)
  Epoch 2 | Step 600/1459 | loss 0.4282 (hard 0.1640, soft 0.6925)
  Epoch 2 | Step 800/1459 | loss 0.4169 (hard 0.1619, soft 0.6719)
  Epoch 2 | Step 1000/1459 | loss 0.4070 (hard 0.1601, s

In [4]:
!pip uninstall -y codebleu tree-sitter tree-sitter-java tree-sitter-python -q
!pip install -q codebleu
!pip install -q "tree-sitter-java==0.21.0"

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [5]:
from codebleu import calc_codebleu

# ── Exact Match ────────────────────────────────────────────────────────────────
exact_match = 100 * sum(
    p.strip() == r.strip() for p, r in zip(predictions, references)
) / len(references)

# ── CodeBLEU ───────────────────────────────────────────────────────────────────
result = calc_codebleu(
    references=[[r] for r in references],
    predictions=predictions,
    lang="java",
    weights=(0.25, 0.25, 0.25, 0.25),
    tokenizer=None
)

# ── Print ──────────────────────────────────────────────────────────────────────
print('\n' + '─' * 50)
print('Evaluation Results')
print('─' * 50)
print(f'  SacreBLEU  : {bleu_result.score:.4f}')
print(f'  ROUGE-L    : {(avg_rouge_l) * 100:.4f}')
print(f'  Exact Match: {exact_match:.4f}')

print('\n' + '─' * 50)
print('CodeBLEU Breakdown')

print('─' * 50)
print(f'  CodeBLEU           : {result["codebleu"]*100:.4f}')
print(f'  N-gram match       : {result["ngram_match_score"]*100:.4f}')
print(f'  Weighted n-gram    : {result["weighted_ngram_match_score"]*100:.4f}')
print(f'  Syntax (AST) match : {result["syntax_match_score"]*100:.4f}')
print(f'  Dataflow match     : {result["dataflow_match_score"]*100:.4f}')
print('─' * 50)


──────────────────────────────────────────────────
Evaluation Results
──────────────────────────────────────────────────
  SacreBLEU  : 79.1775
  ROUGE-L    : 84.6317
  Exact Match: 20.7198

──────────────────────────────────────────────────
CodeBLEU Breakdown
──────────────────────────────────────────────────
  CodeBLEU           : 72.7465
  N-gram match       : 75.0868
  Weighted n-gram    : 75.3045
  Syntax (AST) match : 66.6796
  Dataflow match     : 73.9152
──────────────────────────────────────────────────
